# Camada Gold: Agregações e KPIs de Negócio
Este notebook lê os dados tratados da camada **Silver** e cria tabelas agregadas e views analíticas na camada **Gold**.

**Objetivo:** Facilitar a conexão com ferramentas de Power BI/Tableau e fornecer métricas rápidas.

## Importando as bibliotecas


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from pyspark.sql import functions as F
from pyspark.sql import Window as W
from pyspark.sql.functions import regexp_replace, col, initcap, lower, trim, when, lit, StringType, count, isnan, upper, translate
from pyspark.sql.types import DecimalType, BooleanType
from functools import reduce

## Definição de catalogos e databases do ambiente

In [0]:
%sql
USE CATALOG projeto;
DROP DATABASE gold CASCADE; -- Executar se tiver uma gold já criada
CREATE DATABASE IF NOT EXISTS gold;

---
# 1. Tabelas Agregadas (Físicas)
Tabelas Delta otimizadas para performance em dashboards.

### Tabela Gold: `gold.fato_chamados`
Coisas q ela faz:
- Agregação das tabelas com relacionamento (1:1):
  - silver.fato_chamados
  - silver.dim_custos
  - silver.dim_pesquisa_satisfacao
  - silver.dim_custos

In [0]:
# Fato
df_silver_fato_chamados = spark.table("silver.fato_chamados")

# Dimensões
df_dim_chamados_data = spark.table("silver.dim_chamados_data")
df_dim_pesquisa_satisfacao = spark.table("silver.dim_pesquisa_satisfacao")
df_dim_custos = spark.table("silver.dim_custos")

df_gold = (
    df_silver_fato_chamados
        # Join com dimensão de datas
        .join(
            df_dim_chamados_data,
            on=df_silver_fato_chamados["id_chamado"] == df_dim_chamados_data["id_chamado"],
            how="inner"
        )
        # Join com dimensão de pesquisa de satisfação
        .join(
            df_dim_pesquisa_satisfacao,
            on=df_silver_fato_chamados["id_chamado"] == df_dim_pesquisa_satisfacao["id_chamado"],
            how="left"
        )
        # Join com dimensão de custos
        .join(
            df_dim_custos,
            on=df_silver_fato_chamados["id_chamado"] == df_dim_custos["id_chamado"],
            how="left"
        )
        .select(
            # Todas da fato
            df_silver_fato_chamados["*"],

            # Somente colunas específicas da dimensão de datas
            df_dim_chamados_data["data_hora_abertura"],
            df_dim_chamados_data["data_hora_inicio_atendimento"],
            df_dim_chamados_data["data_hora_finalizacao_atendimento"],
            df_dim_chamados_data["tempo_espera_atendimento_min"],
            df_dim_chamados_data["tempo_atendimento_min"],
            df_dim_chamados_data["ano"],
            df_dim_chamados_data["mes"],
            df_dim_chamados_data["dia"],
            df_dim_chamados_data["dia_semana_num"],
            df_dim_chamados_data["nome_dia"],
            df_dim_chamados_data["trimestre"],
            df_dim_chamados_data["semana_do_ano"],
            df_dim_chamados_data["flag_fim_de_semana"],
            df_dim_chamados_data["ciclo_operacional"],
            df_dim_chamados_data["evento_sazonal"],

            # Colunas específicas das outras dimensões
            df_dim_pesquisa_satisfacao["nota_atendimento"],
            df_dim_custos["valor_custo"]
        )
)

df_gold = df_gold.withColumn(
    "hora_dia",
    F.hour(F.col("data_hora_inicio_atendimento"))
)

df_gold.write.mode("overwrite").saveAsTable("gold.fato_chamados")

display(df_gold.limit(10))

## Tabela Gold: `gold.table_kpis_por_horario`

- Join das tabelas gold.fato_chamados com silver.dim_clientes
- Agrupamento dos dados em relação ao horário gerando informação para cada uma das 24 horas do dia

In [0]:
# Carrega a fato gold
df_gold = spark.table("gold.fato_chamados")

# Carrega a dimensão de clientes (apenas para idade)
df_dim_clientes = spark.table("silver.dim_clientes")

# Join para trazer idade (canal já vem da fato)
df_gold_clientes = (
    df_gold
        .join(df_dim_clientes, on="id_cliente", how="left")
)

# 1) KPIs por horário (24h)
df_kpis_por_horario = (
    df_gold_clientes
        .groupBy("hora_dia")
        .agg(
            F.count("*").alias("qtd_chamados"),
            F.avg("nota_atendimento").alias("nota_media"),
            F.sum("valor_custo").alias("custo_total"),
            F.avg("valor_custo").alias("custo_medio_por_chamado"),
            F.avg("idade").alias("idade_media_cliente")
        )
)

# 2) Canal mais usado por horário
df_canal_por_hora = (
    df_gold_clientes
        .groupBy("hora_dia", "nome_canal")
        .agg(
            F.count("*").alias("qtd_chamados_canal")
        )
)

w = W.partitionBy("hora_dia").orderBy(F.col("qtd_chamados_canal").desc())

df_top_canal_por_hora = (
    df_canal_por_hora
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .select(
            "hora_dia",
            F.col("nome_canal").alias("canal_mais_usado")
        )
)

# 3) Junta KPIs + canal mais usado
df_kpis_por_horario = (
    df_kpis_por_horario
        .join(df_top_canal_por_hora, on="hora_dia", how="left")
        .orderBy(F.col("hora_dia").asc())
)

# Salva como view/tabela na camada gold
df_kpis_por_horario.write.mode("overwrite").saveAsTable(
    "gold.table_kpis_por_horario"
)

display(df_kpis_por_horario)

## Tabela Gold: `gold.table_kpis_dia_semana`

- Join das tabelas gold.fato_chamados com silver.dim_clientes
- Agrupamento dos dados em relação ao dia da semanaa gerando informação para cada um dos sete dias da semana

In [0]:
# Carrega a fato gold
df_gold = spark.table("gold.fato_chamados")

# Carrega a dimensão de clientes (apenas para idade)
df_dim_clientes = spark.table("silver.dim_clientes")

# KPIs por dia da semana
df_kpis_por_dia_semana = (
    df_gold_clientes
        .groupBy("dia_semana_num", "nome_dia")
        .agg(
            F.count("*").alias("qtd_chamados"),
            F.avg("nota_atendimento").alias("nota_media"),
            F.sum("valor_custo").alias("custo_total"),
            F.avg("valor_custo").alias("custo_medio_por_chamado"),
            F.avg("idade").alias("idade_media_cliente")
        )
)

# Canal mais usado por dia da semana
df_canal_por_dia_semana = (
    df_gold_clientes
        .groupBy("dia_semana_num", "nome_canal")
        .agg(
            F.count("*").alias("qtd_chamados_canal")
        )
)

w_dia = W.partitionBy("dia_semana_num").orderBy(F.col("qtd_chamados_canal").desc())

df_top_canal_por_dia_semana = (
    df_canal_por_dia_semana
        .withColumn("rn", F.row_number().over(w_dia))
        .filter(F.col("rn") == 1)
        .select(
            "dia_semana_num",
            F.col("nome_canal").alias("canal_mais_usado")
        )
)

# Junta KPIs + canal mais usado
df_kpis_por_dia_semana = (
    df_kpis_por_dia_semana
        .join(df_top_canal_por_dia_semana, on="dia_semana_num", how="left")
        .orderBy(F.col("dia_semana_num").asc())
)

df_kpis_por_dia_semana.write.mode("overwrite").saveAsTable(
    "gold.table_kpis_dia_semana"
)

display(df_kpis_por_dia_semana)

## Tabela Gold: gold.table_kpis_semana_mes

- Join das tabelas gold.fato_chamados com silver.dim_clientes
- Agrupamento dos dados em relação a cada semana de cada mês

In [0]:
# Carrega a fato gold
df_gold = spark.table("gold.fato_chamados")

# Carrega a dimensão de clientes (apenas para idade)
df_dim_clientes = spark.table("silver.dim_clientes")

# Adiciona coluna de semana do mês (1 a 5) baseada na data de início do atendimento
df_semana_mes = (
    df_gold_clientes
        .withColumn(
            "semana_mes",
            (
                (F.dayofmonth("data_hora_inicio_atendimento") - F.lit(1)) / F.lit(7)
            ).cast("int") + F.lit(1)
        )
)

# KPIs por ano, mês e semana do mês
df_kpis_por_semana_mes = (
    df_semana_mes
        .groupBy("ano", "mes", "semana_mes")
        .agg(
            F.count("*").alias("qtd_chamados"),
            F.avg("nota_atendimento").alias("nota_media"),
            F.sum("valor_custo").alias("custo_total"),
            F.avg("valor_custo").alias("custo_medio_por_chamado"),
            F.avg("idade").alias("idade_media_cliente")
        )
)

# Canal mais usado por ano, mês e semana do mês
df_canal_por_semana_mes = (
    df_semana_mes
        .groupBy("ano", "mes", "semana_mes", "nome_canal")
        .agg(
            F.count("*").alias("qtd_chamados_canal")
        )
)

w_semana_mes = W.partitionBy("ano", "mes", "semana_mes") \
                     .orderBy(F.col("qtd_chamados_canal").desc())

df_top_canal_por_semana_mes = (
    df_canal_por_semana_mes
        .withColumn("rn", F.row_number().over(w_semana_mes))
        .filter(F.col("rn") == 1)
        .select(
            "ano",
            "mes",
            "semana_mes",
            F.col("nome_canal").alias("canal_mais_usado")
        )
)

# Junta KPIs + canal mais usado
df_kpis_por_semana_mes = (
    df_kpis_por_semana_mes
        .join(
            df_top_canal_por_semana_mes,
            on=["ano", "mes", "semana_mes"],
            how="left"
        )
        .orderBy("ano", "mes", "semana_mes")
)

df_kpis_por_semana_mes.write.mode("overwrite").saveAsTable(
    "gold.table_kpis_semana_mes"
)

display(df_kpis_por_semana_mes)

## Tabela Gold: gold.table_kpis_mes

- Join das tabelas gold.fato_chamados com silver.dim_clientes
- Agrupamento dos dados em relação a cada mês

In [0]:
# Carrega a fato gold
df_gold = spark.table("gold.fato_chamados")

# Carrega a dimensão de clientes (apenas para idade)
df_dim_clientes = spark.table("silver.dim_clientes")

# KPIs por ano e mês do ano
df_kpis_por_mes = (
    df_gold_clientes
        .groupBy("ano", "mes")
        .agg(
            F.count("*").alias("qtd_chamados"),
            F.avg("nota_atendimento").alias("nota_media"),
            F.sum("valor_custo").alias("custo_total"),
            F.avg("valor_custo").alias("custo_medio_por_chamado"),
            F.avg("idade").alias("idade_media_cliente")
        )
)

# Canal mais usado por ano e mês
df_canal_por_mes = (
    df_gold_clientes
        .groupBy("ano", "mes", "nome_canal")
        .agg(
            F.count("*").alias("qtd_chamados_canal")
        )
)

w_mes = W.partitionBy("ano", "mes") \
              .orderBy(F.col("qtd_chamados_canal").desc())

df_top_canal_por_mes = (
    df_canal_por_mes
        .withColumn("rn", F.row_number().over(w_mes))
        .filter(F.col("rn") == 1)
        .select(
            "ano",
            "mes",
            F.col("nome_canal").alias("canal_mais_usado")
        )
)

# Junta KPIs + canal mais usado
df_kpis_por_mes = (
    df_kpis_por_mes
        .join(df_top_canal_por_mes, on=["ano", "mes"], how="left")
        .orderBy("ano", "mes")
)

df_kpis_por_mes.write.mode("overwrite").saveAsTable(
    "gold.table_kpis_mes"
)

display(df_kpis_por_mes)

### Tabela Gold: `gold.desempenho_canal`
Coisas q ela faz:
- Consolida a visão de eficiência técnica e financeira dos canais.
  - Agregação das tabelas com relacionamento:
  - silver.dim_pesquisa_satisfacao
  - silver.dim_canais (Nome do canal)
  - silver.dim_custos (Financeiro)
  - silver.dim_pesquisa_satisfacao (Qualidade)
  - silver.dim_motivos (Criticidade e Contexto)

  

In [0]:
# Carregando as tabelas
df_chamados = spark.table("projeto.silver.fato_chamados")
df_canais   = spark.table("projeto.silver.dim_canais")
df_custos   = spark.table("projeto.silver.dim_custos")
df_pesquisa = spark.table("projeto.silver.dim_pesquisa_satisfacao")
df_motivos  = spark.table("projeto.silver.dim_motivos")

df_join = (
    df_chamados.alias("c")
        .join(
            df_canais.alias("can"),
            F.lower(F.col("c.nome_canal")) == F.lower(F.col("can.nome_canal")),
            how="left"
        )
        .join(
            df_custos.alias("fin"),
            F.col("c.id_chamado") == F.col("fin.id_chamado"),
            how="left"
        )
        .join(
            df_pesquisa.alias("pesq"),
            F.col("c.id_chamado") == F.col("pesq.id_chamado"),
            how="left"
        )
        .join(
            df_motivos.alias("m"),
            F.lower(F.col("c.nome_motivo_clean")) == F.lower(F.col("m.nome_motivo_clean")),
            how="left"
        )
)

df_gold_desempenho_canal = df_join.select(
    F.col("c.id_chamado"),
    F.col("c.id_cliente"),
    F.col("c.id_atendente"),
    F.col("c.nome_canal"),
    F.col("c.resolvido"),
    F.coalesce(F.col("m.nome_motivo"), F.col("c.nome_motivo_clean")).alias("motivo"),
    F.col("m.categoria_motivo"), 
    F.col("m.criticidade_motivo"),
    F.col("fin.valor_custo"),
    F.col("pesq.nota_atendimento"),
    F.current_timestamp().alias("gold_ingestion_timestamp")
)

df_gold_desempenho_canal.write.mode("overwrite").saveAsTable("gold.fato_desempenho_canal")

display(df_gold_desempenho_canal.limit(10))

### Tabela Gold: `gold.fato_experiencia_cliente`
Coisas q ela faz:
- Cria a visão 360º da jornada do consumidor enriquecida com segmentação.
- Agregação das tabelas com relacionamento
  - silver.fato_chamados (Base)
  - silver.dim_clientes (Perfil demográfico)
  - silver.dim_pesquisa_satisfacao (Nota do cliente)
  - silver.dim_canais (Canal utilizado)

In [0]:
df_chamados = spark.table("projeto.silver.fato_chamados")
df_clientes = spark.table("projeto.silver.dim_clientes")
df_pesquisa = spark.table("projeto.silver.dim_pesquisa_satisfacao")
df_canais   = spark.table("projeto.silver.dim_canais")

df_join_cliente = df_chamados.alias("c") \
    .join(df_clientes.alias("cli"), on="id_cliente", how="left") \
    .join(df_pesquisa.alias("pesq"), on="id_chamado", how="left") \
    .join(df_canais.alias("can"), F.lower(F.col("c.nome_canal")) == F.lower(F.col("can.nome_canal")), how="left")

df_gold_experiencia_cliente = df_join_cliente.select(
    "c.id_chamado",
    "c.id_cliente",
    "cli.nome",
    "cli.faixa_etaria_geracao", 
    "cli.flag_idoso",
    "cli.regiao",
    F.coalesce("can.nome_canal", "c.nome_canal").alias("canal_utilizado"),
    "c.resolvido",
    "pesq.nota_atendimento",
    F.when(F.col("c.resolvido") == True, 1).otherwise(0).alias("flag_sucesso"),
    F.when(F.col("pesq.nota_atendimento").isNotNull(), 1).otherwise(0).alias("flag_respondeu_pesquisa")
)

df_gold_experiencia_cliente.write.mode("overwrite").saveAsTable("gold.fato_experiencia_cliente")
display(df_gold_experiencia_cliente.limit(10))

### Tabela Gold: `gold.dim_motivos_criticos`
Coisas q ela faz:
- Identificação de Gargalos Operacionais
- Segmentação por Categorias
- Agregação das tabelas com relacionamento
  - silver.fato_chamados (base)
  - silver.dim_motivos (categorias + criticidade)
  - silver.dim_pesquisa_satisfacao (CSAT)
  - silver.dim_chamados_data (TME)
  - silver.dim_custos (custo)



In [0]:
df_fato_chamados = spark.table("projeto.silver.fato_chamados")
df_dim_motivos = spark.table("projeto.silver.dim_motivos")
df_dim_custos = spark.table("projeto.silver.dim_custos")
df_dim_pesquisa = spark.table("projeto.silver.dim_pesquisa_satisfacao")
df_dim_chamados_data = spark.table("projeto.silver.dim_chamados_data")

# Calcular valores máximos para normalização
max_values = df_fato_chamados.agg(
    F.count("id_chamado").alias("total_chamados")
).join(
    df_dim_custos.agg(F.max("valor_custo").alias("max_custo")), 
    F.lit(True)
).join(
    df_dim_chamados_data.agg(F.max("tempo_espera_atendimento_min").alias("max_tme")), 
    F.lit(True)
).collect()[0]

max_custo = max_values["max_custo"] or 1
max_tme = max_values["max_tme"] or 1
total_chamados = max_values["total_chamados"] or 1

# 1. Agregar dados básicos de chamados por motivo
chamados_por_motivo = (
    df_fato_chamados
    .groupBy("nome_motivo_clean")
    .agg(F.count("id_chamado").alias("qtd_chamados"))
)

# 2. Criar um mapeamento manual baseado na análise dos dados
df_motivos_mapeamento = df_dim_motivos.select(
    "nome_motivo", 
    "categoria_motivo", 
    "criticidade_motivo"
).withColumn(
    "nome_truncado",
    F.when(F.col("nome_motivo").contains("COMPRA NAO AUTORIZADA"), "COMPRA NO AUTORIZADA")
     .when(F.col("nome_motivo").contains("CONTESTACAO DE FATURA"), "CONTESTAO DE FATURA")
     .when(F.col("nome_motivo").contains("DESBLOQUEIO DE CARTAO"), "DESBLOQUEIO DE CARTO")
     .when(F.col("nome_motivo").contains("CONSULTA DE CONTRATO"), "CONSULTA DE CONTRATO")
     .when(F.col("nome_motivo").contains("CONTRATACAO DE CARTAO ADICIONAL"), "CONTRATAO DE CARTO ADICIONAL")
     .when(F.col("nome_motivo").contains("BLOQUEIO DE CARTAO"), "BLOQUEIO DE CARTO")
     .when(F.col("nome_motivo").contains("PROBLEMA COM APLICATIVO"), "PROBLEMA COM APLICATIVO")
     .when(F.col("nome_motivo").contains("ALTERACAO DE DADOS CADASTRAIS"), "ALTERAO DE DADOS CADASTRAIS (VENCIMENTO DA FATURA, TELEFONE, EMAIL)")
     .when(F.col("nome_motivo").contains("DUVIDAS GERAIS SOBRE PROGRAMA DE PONTOS"), "DVIDAS GERAIS SOBRE PROGRAMA DE PONTOS")
     .when(F.col("nome_motivo").contains("CONSULTA DE LIMITE"), "CONSULTA DE LIMITE")
     .otherwise(F.col("nome_motivo"))
)

# Fazer o join com o mapeamento
motivos_com_info = (
    chamados_por_motivo
    .join(
        df_motivos_mapeamento,
        chamados_por_motivo["nome_motivo_clean"] == df_motivos_mapeamento["nome_truncado"],
        "left"
    )
    .withColumn("categoria_motivo", F.coalesce(F.col("categoria_motivo"), F.lit("NÃO CATEGORIZADO")).cast("string"))
    .withColumn("criticidade_motivo", F.coalesce(F.col("criticidade_motivo"), F.lit("Média")).cast("string"))
    .withColumn("qtd_chamados", F.col("qtd_chamados").cast("long"))
    .drop("nome_truncado")
)

# 3. Agregar CSAT por motivo
csat_por_motivo = (
    df_fato_chamados
    .join(df_dim_pesquisa, "id_chamado", "left")
    .groupBy("nome_motivo_clean")
    .agg(F.avg("nota_atendimento").alias("csat_medio"))
    .withColumn("csat_medio", F.round(F.col("csat_medio"), 2).cast("double"))
)

# 4. Agregar TME por motivo
tme_por_motivo = (
    df_fato_chamados
    .join(df_dim_chamados_data.select("id_chamado", "tempo_espera_atendimento_min"), "id_chamado", "left")
    .groupBy("nome_motivo_clean")
    .agg(F.avg("tempo_espera_atendimento_min").alias("tme_medio"))
    .withColumn("tme_medio", F.round(F.col("tme_medio"), 2).cast("double"))
)

# 5. Agregar custo por motivo
custo_por_motivo = (
    df_fato_chamados
    .join(df_dim_custos, "id_chamado", "left")
    .groupBy("nome_motivo_clean")
    .agg(F.sum("valor_custo").alias("custo_total"))
    .withColumn("custo_total", F.round(F.col("custo_total"), 2).cast("double"))
)

# 6. Consolidar todas as métricas
df_consolidado = (
    motivos_com_info
    .join(csat_por_motivo, "nome_motivo_clean", "left")
    .join(tme_por_motivo, "nome_motivo_clean", "left")
    .join(custo_por_motivo, "nome_motivo_clean", "left")
)

# 7. Calcular scores de prioridade
df_com_scores = df_consolidado.withColumn(
    "score_custo", 
    F.when(F.col("custo_total").isNull(), 0).otherwise(F.col("custo_total") / F.lit(max_custo))
).withColumn(
    "score_volume", 
    F.col("qtd_chamados") / F.lit(total_chamados)
).withColumn(
    "score_tme",
    F.when(F.col("tme_medio").isNull(), 0).otherwise(F.col("tme_medio") / F.lit(max_tme))
).withColumn(
    "score_csat",
    F.when(F.col("csat_medio").isNull(), 1).otherwise(1 - (F.col("csat_medio") / 5.0))
)

# 8. Calcular score final e ranking
df_final = df_com_scores.withColumn(
    "score_prioridade",
    (F.col("score_custo") * 0.3) + 
    (F.col("score_volume") * 0.25) + 
    (F.col("score_tme") * 0.25) + 
    (F.col("score_csat") * 0.20)
)

# Usar window function apenas para o ranking
df_final = df_final.withColumn(
    "ranking_prioridade", 
    F.row_number().over(W.orderBy(F.col("score_prioridade").desc()))
)

# 9. Selecionar apenas os campos solicitados
df_gold_dim_motivos_criticos = df_final.select(
    F.col("nome_motivo_clean").alias("motivo").cast("string"),
    F.col("categoria_motivo").cast("string"),
    F.col("criticidade_motivo").cast("string"),
    F.col("qtd_chamados").cast("long"),
    F.col("csat_medio").cast("double"),
    F.col("tme_medio").cast("double"),
    F.col("custo_total").cast("double"),
    F.col("ranking_prioridade").cast("int")
).orderBy("ranking_prioridade")

display(df_gold_dim_motivos_criticos)

df_gold_dim_motivos_criticos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("projeto.gold.dim_motivos_criticos")

print("\n DISTRIBUIÇÃO FINAL POR CATEGORIA:")
df_gold_dim_motivos_criticos.groupBy("categoria_motivo").agg(
    F.count("*").alias("qtd_motivos"),
    F.avg("ranking_prioridade").alias("ranking_medio"),
    F.sum("qtd_chamados").alias("total_chamados")
).orderBy("total_chamados", ascending=False).show()

### View Gold: `gold.view_desempenho_atendente`
A view proporciona uma visão sobre o desempenho dos atendentes. Ela permite avaliar cada atendente em vários aspectos importantes, como:

- Volume de trabalho → quantidade total de atendimentos (qtd_atendimentos)
- Produtividade (tempo médio e tempo total trabalhado)
- Eficiência (taxa de resolução de chamados)
- Qualidade (nota média recebida pelos atendentes)
- Nível do atendente e descrição do nível
- Custo operacional (custo médio associado aos atendimentos)

Em resumo, essa view transforma dados dispersos em uma visão unificada de performance, qualidade e custo por atendente.

In [0]:
%sql
CREATE OR REPLACE VIEW gold.view_desempenho_atendentes AS
SELECT
    a.id_atendente,
    a.nome_atendente,
    a.nivel_atendimento,
    a.descricao_nivel,

    COUNT(c.id_chamado) AS qtd_atendimentos,

    ROUND(AVG(c.tempo_atendimento_min), 2) AS tempo_medio_atendimento_min,
    ROUND(SUM(c.tempo_atendimento_min), 2) AS tempo_total_trabalhado_min,

    ROUND(
        SUM(CASE WHEN c.resolvido = TRUE THEN 1 ELSE 0 END) 
        / COUNT(c.id_chamado) * 100, 
        2
    ) AS taxa_resolucao,

    ROUND(AVG(c.nota_atendimento), 2) AS nota_media,

    ROUND(
        CAST(AVG(c.valor_custo) AS DECIMAL(26, 18)),2) AS custo_medio_atendente,
    
    ROUND(SUM(c.valor_custo), 2) AS custo_total_atendente

FROM gold.fato_chamados c
RIGHT JOIN silver.dim_atendentes a
    ON c.id_atendente = a.id_atendente

GROUP BY
    a.id_atendente,
    a.nome_atendente,
    a.nivel_atendimento,
    a.descricao_nivel;


In [0]:
%sql
SELECT * 
FROM gold.view_desempenho_atendentes
LIMIT 10;

### View Gold: `Views sobre os custos relacionados a cada motivo e a cada nível de atendimento`
Objetivo de facilitar a visualização das áreas mais custosas:
- Por motivo de chamados
- E por cada nivel de atendimento

## View Gold: `gold.view_analise_custos_motivos`

In [0]:
query_custos_motivo = """
CREATE OR REPLACE VIEW gold.view_analise_custos_motivos AS
SELECT 
    CASE 
        WHEN fato_chamados.nome_motivo_clean = 'COMPRA NO AUTORIZADA' THEN 'Compra Não Autorizada'
        WHEN fato_chamados.nome_motivo_clean = 'CONTESTAO DE FATURA' THEN 'Contestação de Fatura'
        WHEN fato_chamados.nome_motivo_clean = 'DESBLOQUEIO DE CARTO' THEN 'Desbloqueio de Cartão'
        WHEN fato_chamados.nome_motivo_clean = 'CONSULTA DE CONTRATO' THEN 'Consulta de Contrato'
        WHEN fato_chamados.nome_motivo_clean = 'CONTRATAO DE CARTO ADICIONAL' THEN 'Contratação de Cartão Adicional'
        WHEN fato_chamados.nome_motivo_clean = 'BLOQUEIO DE CARTO' THEN 'Bloqueio de Cartão'
        WHEN fato_chamados.nome_motivo_clean = 'PROBLEMA COM APLICATIVO' THEN 'Problema com Aplicativo'
        WHEN fato_chamados.nome_motivo_clean LIKE 'ALTERAO DE DADOS%' THEN 'Alteração de Dados Cadastrais'
        WHEN fato_chamados.nome_motivo_clean = 'DVIDAS GERAIS SOBRE PROGRAMA DE PONTOS' THEN 'Dúvidas Gerais sobre Programa de Pontos'
        WHEN fato_chamados.nome_motivo_clean = 'CONSULTA DE LIMITE' THEN 'Consulta de Limite'
        WHEN fato_chamados.nome_motivo_clean = 'CONSULTA DE FATURA' THEN 'Consulta de Fatura'
        ELSE initcap(fato_chamados.nome_motivo_clean) -- Caso apareça algum novo, deixa apenas a primeira letra maiúscula
    END AS motivo,
    m.categoria_motivo,
    COUNT(id_chamado) AS volume_chamados,
    CAST(SUM(valor_custo) AS DECIMAL(10,2)) AS custo_total,
    CAST(AVG(valor_custo) AS DECIMAL(10,2)) AS custo_medio_por_chamado,
    CAST(AVG(tempo_atendimento_min) AS DECIMAL(10,2)) AS tma_medio,
    ROUND((SUM(valor_custo) / SUM(SUM(valor_custo)) OVER()) * 100, 2) AS pct_custo_total

FROM gold.fato_chamados join silver.dim_motivos m ON fato_chamados.nome_motivo_clean = m.nome_motivo_clean
GROUP BY fato_chamados.nome_motivo_clean, m.categoria_motivo
ORDER BY custo_total DESC
"""

spark.sql(query_custos_motivo)

In [0]:
%sql
SELECT * 
FROM gold.view_analise_custos_motivos
LIMIT 10;

## View Gold: `gold.view_analise_custos_regiao`

In [0]:
query_custos_regiao = """
CREATE OR REPLACE VIEW gold.view_analise_custos_regiao AS
SELECT 
    -- Tratamento para garantir que não venham nulos ou nomes despadronizados
    COALESCE(c.regiao, 'Não Informado') AS regiao,
    
    -- Métricas de Volumetria e Custo
    COUNT(f.id_chamado) AS volume_chamados,
    CAST(SUM(f.valor_custo) AS DECIMAL(10,2)) AS custo_total,
    CAST(AVG(f.valor_custo) AS DECIMAL(10,2)) AS custo_medio_por_chamado,
    CAST(AVG(f.tempo_atendimento_min) AS DECIMAL(10,2)) AS tma_medio,
    
    -- Percentual representativo do custo total (Pareto)
    ROUND((SUM(f.valor_custo) / SUM(SUM(f.valor_custo)) OVER()) * 100, 2) AS pct_custo_total

FROM gold.fato_chamados f
-- Join com a tabela de clientes para pegar a Região
LEFT JOIN silver.dim_clientes c ON f.id_cliente = c.id_cliente

GROUP BY 1
ORDER BY custo_total DESC
"""

spark.sql(query_custos_regiao)

In [0]:
%sql
SELECT * 
FROM gold.view_analise_custos_regiao
LIMIT 10;

### View Gold: `gold.view_canais`
Coisas q ela faz:
permitem que a diretoria monitore a eficiência técnica (quem resolve mais), prove o retorno financeiro da automação (comparando o custo irrisório do Robô vs. Humano) e identifique gargalos de UX, apontando exatamente em quais tipos de problema a tecnologia falha e frustra o cliente. É uma visão completa de Custo, Qualidade e Resolução.

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW gold.view_canais AS SELECT 
    nome_canal AS canal,
    COUNT(id_chamado) AS volume_chamados,
    ROUND((SUM(CASE WHEN resolvido = true THEN 1 ELSE 0 END) / COUNT(id_chamado)) * 100, 2) AS taxa_resolucao_pct,
    ROUND(100 - ((SUM(CASE WHEN resolvido = true THEN 1 ELSE 0 END) / COUNT(id_chamado)) * 100), 2) AS taxa_potencial_transferencia_pct,
    ROUND(AVG(nota_atendimento), 2) AS csat_medio,
    CAST(AVG(tempo_atendimento_min) AS DECIMAL(10,2)) AS tma_medio,
    CAST(SUM(valor_custo) AS DECIMAL(10,2)) AS custo_total,
    CAST(AVG(valor_custo) AS DECIMAL(10,2)) AS custo_medio_por_chamado,
    ROUND((SUM(valor_custo) / SUM(SUM(valor_custo)) OVER()) * 100, 2) AS pct_custo_total

FROM gold.fato_chamados
GROUP BY nome_canal
ORDER BY custo_total DESC
""")


In [0]:
%sql
SELECT * 
FROM gold.view_canais
LIMIT 10;

## View Gold: `gold.view_comparativo_humano_robo`

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW projeto.gold.view_comparativo_humano_robo AS
    SELECT 
        CASE 
            WHEN lower(nome_canal) IN ('chatbot', 'ura', 'whatsapp bot', 'app', 'site') THEN 'Robô/Digital'
            ELSE 'Atendimento Humano' 
        END AS tipo_atendimento,
        COUNT(id_chamado) AS volumetria,
        COUNT(nota_atendimento) AS qtd_pesquisas_respondidas,
        ROUND(AVG(nota_atendimento), 2) AS csat_medio,
        ROUND(SUM(CASE WHEN resolvido = true THEN 1 ELSE 0 END) / COUNT(id_chamado) * 100, 2) AS taxa_resolucao_pct,
        ROUND(SUM(valor_custo), 2) AS custo_total_acumulado,
        ROUND(AVG(valor_custo), 4) AS custo_medio_unitario
        
    FROM projeto.gold.fato_desempenho_canal
    GROUP BY 1
""")

In [0]:
%sql
SELECT * 
FROM gold.view_comparativo_humano_robo
LIMIT 10;     

## View Gold: `gold.view_diagnostico_bots`

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW projeto.gold.view_diagnostico_bots AS
    SELECT 
        nome_canal,
        criticidade_motivo AS complexidade,
        COUNT(id_chamado) AS tentativas,
        ROUND(SUM(CASE WHEN resolvido = true THEN 1 ELSE 0 END) / COUNT(id_chamado) * 100, 1) AS taxa_sucesso_pct,
        ROUND(AVG(nota_atendimento), 1) AS satisfacao_usuario
        
    FROM projeto.gold.fato_desempenho_canal
    WHERE upper(nome_canal) IN ('URA', 'CHATBOT', 'ATENDIMENTO INICIAL', 'ATENDIMENTO ESPECIALIZADO')
    GROUP BY nome_canal, criticidade_motivo
    ORDER BY nome_canal, tentativas DESC
""")

In [0]:
%sql
SELECT * 
FROM gold.view_diagnostico_bots
LIMIT 10;

## View Gold: `gold.view_analise_geracoes_tech`


In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW projeto.gold.view_analise_geracoes_tech AS
    SELECT 
        faixa_etaria_geracao,
        canal_utilizado,
        COUNT(id_chamado) AS total_interacoes,
        ROUND(AVG(nota_atendimento), 2) AS csat_medio,
        ROUND(SUM(flag_sucesso) / COUNT(id_chamado) * 100, 2) AS taxa_resolucao_pct,
        SUM(flag_respondeu_pesquisa) AS qtd_pesquisas_respondidas
        
    FROM projeto.gold.fato_experiencia_cliente
    GROUP BY faixa_etaria_geracao, canal_utilizado
    ORDER BY faixa_etaria_geracao, total_interacoes DESC
""")


In [0]:
%sql
SELECT * 
FROM gold.view_analise_geracoes_tech
LIMIT 10;

## View Gold: `gold.view_jornada_cliente_360`

In [0]:

spark.sql("""
    CREATE OR REPLACE VIEW projeto.gold.view_jornada_cliente_360 AS
    WITH ranking_canais AS (
        SELECT 
            id_cliente, 
            canal_utilizado, 
            COUNT(*) as frequencia,
            ROW_NUMBER() OVER(PARTITION BY id_cliente ORDER BY COUNT(*) DESC) as rank
        FROM projeto.gold.fato_experiencia_cliente
        GROUP BY id_cliente, canal_utilizado
    )
    SELECT 
        base.id_cliente,
        base.nome,
        base.faixa_etaria_geracao,
        base.regiao,
        rc.canal_utilizado AS canal_mais_usado,
        COUNT(base.id_chamado) AS qtd_chamados_total,
        SUM(CASE WHEN base.resolvido = false THEN 1 ELSE 0 END) AS qtd_falhas_resolucao,
        ROUND(SUM(base.flag_sucesso) / COUNT(base.id_chamado) * 100, 1) AS taxa_sucesso_pessoal_pct,
        ROUND(AVG(base.nota_atendimento), 2) AS csat_medio_pessoal
    FROM projeto.gold.fato_experiencia_cliente base
    LEFT JOIN ranking_canais rc ON base.id_cliente = rc.id_cliente AND rc.rank = 1
    GROUP BY 
        base.id_cliente, 
        base.nome, 
        base.faixa_etaria_geracao, 
        base.regiao, 
        rc.canal_utilizado
""")

In [0]:
%sql
SELECT * 
FROM gold.view_jornada_cliente_360
LIMIT 10;

## View Gold: `gold.view_kpi_criticidade`

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold.view_kpi_criticidade AS
WITH base AS (
    SELECT
        dm.criticidade_motivo,

        COUNT(*)                                                  AS qtd_chamados,
        ROUND(AVG(f.nota_atendimento), 2)                         AS nota_media,
        ROUND(AVG(f.tempo_espera_atendimento_min), 2)             AS tempo_espera_medio_min,
        ROUND(AVG(f.tempo_atendimento_min), 2)                    AS tempo_atendimento_medio_min,
        ROUND(SUM(f.valor_custo), 2)                              AS custo_total,
        ROUND(AVG(f.valor_custo), 2)                              AS custo_medio_por_chamado,
        ROUND(AVG(c.idade), 2)                                    AS idade_media,

        SUM(CAST(f.resolvido AS INT))                             AS qtd_resolvidos,

        -- Quantidade de chamados por nível
        SUM(CASE WHEN a.nivel_atendimento = 1 THEN 1 ELSE 0 END)  AS qtd_chamados_nivel_1,
        SUM(CASE WHEN a.nivel_atendimento = 2 THEN 1 ELSE 0 END)  AS qtd_chamados_nivel_2,

        -- Quantidade de chamados resolvidos por nível
        SUM(
            CASE WHEN a.nivel_atendimento = 1
                 THEN CAST(f.resolvido AS INT)
                 ELSE 0
            END
        ) AS qtd_resolvidos_nivel_1,

        SUM(
            CASE WHEN a.nivel_atendimento = 2
                 THEN CAST(f.resolvido AS INT)
                 ELSE 0
            END
        ) AS qtd_resolvidos_nivel_2

    FROM gold.fato_chamados            AS f
    LEFT JOIN silver.dim_motivos       AS dm ON f.nome_motivo_clean = dm.nome_motivo_clean
    LEFT JOIN silver.dim_clientes      AS c  ON f.id_cliente        = c.id_cliente
    LEFT JOIN silver.dim_atendentes    AS a  ON f.id_atendente      = a.id_atendente

    GROUP BY dm.criticidade_motivo
)

SELECT
    criticidade_motivo,
    qtd_chamados,
    nota_media,
    tempo_espera_medio_min,
    tempo_atendimento_medio_min,
    custo_total,
    custo_medio_por_chamado,
    idade_media,
    qtd_resolvidos,

    -- Já existente
    (qtd_chamados - qtd_resolvidos)                             AS qtd_nao_resolvidos,
    ROUND(qtd_resolvidos        / qtd_chamados * 100, 2)        AS perc_resolvidos,
    ROUND((qtd_chamados - qtd_resolvidos) / qtd_chamados * 100, 2) AS perc_nao_resolvidos,

    -- Por nível
    qtd_chamados_nivel_1,
    qtd_chamados_nivel_2,
    qtd_resolvidos_nivel_1,
    qtd_resolvidos_nivel_2,

    (qtd_chamados_nivel_1 - qtd_resolvidos_nivel_1)             AS qtd_nao_resolvidos_nivel_1,
    (qtd_chamados_nivel_2 - qtd_resolvidos_nivel_2)             AS qtd_nao_resolvidos_nivel_2,

    ROUND(qtd_resolvidos_nivel_1 / qtd_chamados_nivel_1 * 100, 2) AS taxa_resolucao_nivel_1,
    ROUND(qtd_resolvidos_nivel_2 / qtd_chamados_nivel_2 * 100, 2) AS taxa_resolucao_nivel_2,

    ROUND(qtd_chamados_nivel_1 / qtd_chamados * 100, 2)         AS perc_chamados_nivel_1,
    ROUND(qtd_chamados_nivel_2 / qtd_chamados * 100, 2)         AS perc_chamados_nivel_2

FROM base
ORDER BY custo_medio_por_chamado
""")

In [0]:
%sql
SELECT * 
FROM gold.view_kpi_criticidade
LIMIT 10;

# Validação Final
Listando todas as tabelas e views criadas no database Gold.

In [0]:
display(spark.sql("SHOW TABLES IN gold"))